#### coauthor_extras_exante.ipynb

**Author:** James Sayre
**Email:** jsayre@ucdavis.edu
**Date Modified:** 2026-05-29

**Description:** Ex-ante predictability of farm-level (ADC) accuracy from
*municipality-level* information only (no downscaled ground truth). Consolidates
the standalone analysis scripts that previously lived in `/tmp` into one
documented, version-controlled notebook. Four sections:

- **A. Baseline predictors** — 9 mun-level features (SIAP, geometry, model
  self-predictions, AEF heterogeneity) vs three targets: within-mun R², per-mun
  contribution to pooled total R², and mun-mean level error.
- **B. Enriched model + trust index** — adds 7 within-heterogeneity features
  (irrigation gradient, area CV, AEF covariance spread / PC1 share / effective
  dimension); builds the CV-predicted within-R² "trust index".
- **C. Trust-index choropleth** — ex-ante predicted vs actual within-R² maps.
- **D. Push (target-noise diagnostic)** — shows the apparent ceiling on
  within-skill predictability was largely *target noise*: on well-estimated muns
  (n_adc ≥ 20–30) a per-mun Spearman-ρ predictor reaches holdout R² ≈ 0.44 and a
  positive-skill classifier reaches AUC ≈ 0.81. Adds multi-year AEF spread and an
  interpretable `irrig_share_sd × log_n_adc` interaction (p = 0.001).

**Environment note:** Run in `geo_env` (`~/miniforge3/envs/geo_env/bin/python`),
which has both scikit-learn (needed for A/B/D) and geopandas (needed for C).
Each cell is self-contained (defines its own paths) and re-derives the per-mun
targets, so cells can be run independently. Section C consumes
`exante_trust_index.csv` from Section B.

**Inputs:**
- `~/Dropbox/Projects/Maize_prediction/Data/predictions/adc_aef_hist_ens_eval.parquet` (headline eval; muncode, adc, yield, pred — INEGI 2022)
- `~/Dropbox/Projects/Maize_prediction/Data/alpha_earth/alpha_earth_mex_adcs.parquet` (64 AEF dims A00–A63 × adcid × year)
- `~/Dropbox/Projects/Maize_prediction/Data/SIAP_agland/Output/2007_adcs_agland_area.csv` (ADC irrigation / area / maize-land)
- `~/Dropbox/Projects/The Promise of Crop Substitution/data/SIAP/Cleaned/siap_ag_prod_estimation_ca2007.dta` (SIAP maize yields, cross-year)
- `~/Dropbox/Projects/Maize_prediction/plots/coauthor_extras_paper/drivers_mun.csv` (precomputed mun drivers)
- `~/Dropbox/Projects/Maize_prediction/Data/SIAP_agland/Output/agland_mun16.shp` (mun polygons, for the map)
- `~/Dropbox/Projects/Crop_misallocation/Data/Municipality_shp/STATES.shp` (state boundaries)

**Outputs:** (all under `~/Dropbox/Projects/Maize_prediction/plots/coauthor_extras_paper/`)
- `exante_feature_table.csv`, `exante_feature_importance.csv`, `fig_exante_accuracy_predictors.png`
- `exante_trust_index.csv`, `exante_within_importance_enriched.csv`, `fig_exante_within_enriched.png`
- `fig_trust_index_map.png`
- `exante_trust_index_v2.csv`, `fig_exante_samplesize_diagnostic.png`


#### A. Baseline ex-ante predictors (9 features, 3 targets)

Source: `exante_predictors.py`. Builds per-mun targets from the INEGI-2022 eval
frame and predicts them from 9 features computable without ADC-level ground
truth. Run in `ml_cuda`.


In [1]:
"""Ex-ante (mun-level, no ground-truth) predictors of farm-level within/total R^2.

Targets (per mun, from INEGI 2022 validation):
  - within_r2_mun  : R^2 of within-mun ADC deviations (relative ranking skill)
  - total_contrib  : 1 - SS_res_mun / SS_tot_global_mun (per-mun contribution to pooled total R^2)
  - mun_mean_abserr: |mean(pred) - mean(obs)| at mun level (between/level component)

Features (all computable WITHOUT ADC-level ground truth):
  SIAP:     siap_sd_yield (cross-year), siap_mean_yield, irrig_share, maize_share
  Geometry: log_n_adc (count of ALL ADCs in mun, geography), log_mean_ha_adc (mean ha per ADC)
  Model:    sd_pred (within-mun SD of AEF predictions), mean_pred
  Satellite:aef_heterogeneity (mean over 64 dims of within-mun SD of AEF embeddings)
"""
import os, time
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams.update({"font.family":"serif","figure.dpi":130,
                     "axes.titlesize":12,"axes.labelsize":11,
                     "xtick.labelsize":9,"ytick.labelsize":9,"legend.fontsize":9})

home = os.path.expanduser("~")
proj = os.path.join(home, "Dropbox/Projects/Maize_prediction")
data = os.path.join(proj, "Data")
preds_dir = os.path.join(data, "predictions")
out_dir = os.path.join(proj, "plots/coauthor_extras_paper")
siap_pkg = os.path.join(home, "Dropbox/Projects/The Promise of Crop Substitution/data/SIAP/Cleaned")

def r2(y, yh):
    m = np.isfinite(y) & np.isfinite(yh)
    y, yh = np.asarray(y[m]), np.asarray(yh[m])
    if len(y) < 2: return np.nan
    st = np.sum((y - y.mean())**2)
    return 1 - np.sum((y - yh)**2)/st if st > 0 else np.nan

# ---------- Targets from eval frame ----------
df = pd.read_parquet(os.path.join(preds_dir, "adc_aef_hist_ens_eval.parquet"))
df = df[["muncode","adc","yield","pred"]].dropna(subset=["yield","pred"]).copy()
df["muncode"] = df["muncode"].astype(str).str.zfill(5)
global_mean = df["yield"].mean()

rows = []
for mun, g in df.groupby("muncode"):
    if len(g) < 2: continue
    ym, pm = g["yield"].mean(), g["pred"].mean()
    ss_res = float(((g["yield"]-g["pred"])**2).sum())
    ss_tot_within = float(((g["yield"]-ym)**2).sum())
    ss_tot_global = float(((g["yield"]-global_mean)**2).sum())
    within = 1 - ss_res/ss_tot_within if ss_tot_within>0 else np.nan
    total_contrib = 1 - ss_res/ss_tot_global if ss_tot_global>0 else np.nan
    rows.append({"muncode":mun, "n_adc":len(g),
                  "within_r2_mun":within, "total_contrib":total_contrib,
                  "mun_mean_abserr":abs(pm-ym)})
targets = pd.DataFrame(rows)
print(f"Targets for {len(targets):,} muns")

# ---------- Ex-ante features ----------
# 1) from drivers_mun.csv (ex-ante subset)
dm = pd.read_csv(os.path.join(out_dir, "drivers_mun.csv"))
dm["muncode"] = dm["muncode"].astype(str).str.zfill(5)
feat = dm[["muncode","irrig_share","maize_share","sd_pred","mean_pred"]].copy()
# Ex-ante geometry from the FULL ADC partition (all control areas in the mun),
# NOT the maize-eval subset: n_adc_geo = # ADCs per mun, mean_ha_adc = mean ha/ADC
ag_geo = pd.read_csv(os.path.join(data,"SIAP_agland/Output/2007_adcs_agland_area.csv"))
ag_geo["muncode"] = ag_geo["adc07"].astype(str).str.replace("-","",regex=False).str[:5]
geo = ag_geo.groupby("muncode").agg(n_adc_geo=("adc_area","size"), mean_ha_adc=("adc_area","mean")).reset_index()
feat = feat.merge(geo, on="muncode", how="left")

# 2) SIAP cross-year SD + mean (maize, 2007-2024)
siap = pd.read_stata(os.path.join(siap_pkg,"siap_ag_prod_estimation_ca2007.dta"),
                      columns=["year","muncode","name","q","ha_planted"])
siap = siap[(siap["name"]=="Maize") & siap["year"].between(2007,2024)].copy()
siap["yld"] = siap["q"]/siap["ha_planted"].replace(0,np.nan)
siap = siap.dropna(subset=["yld"])
siap["muncode"] = siap["muncode"].astype(str).str.zfill(5)
siap_ag = siap.groupby("muncode").agg(siap_sd_yield=("yld","std"),
                                       siap_mean_yield=("yld","mean"),
                                       siap_nyr=("year","nunique")).reset_index()
siap_ag = siap_ag[siap_ag["siap_nyr"]>=5]
feat = feat.merge(siap_ag[["muncode","siap_sd_yield","siap_mean_yield"]], on="muncode", how="left")

# 3) AEF within-mun heterogeneity (mean over 64 dims of within-mun SD), 2022
print("Loading AEF embeddings (2022)...")
t0=time.time()
aef = pd.read_parquet(os.path.join(data,"alpha_earth/alpha_earth_mex_adcs.parquet"),
                       filters=[("year","==",2022)])
acols = [f"A{i:02d}" for i in range(64)]
aef["muncode"] = aef["adcid"].astype(str).str.replace("-","",regex=False).str[:5]
print(f"  AEF 2022: {len(aef):,} ADCs in {(time.time()-t0)/60:.1f} min")
# within-mun SD per dim, then mean across dims
grp = aef.groupby("muncode")
aef_sd = grp[acols].std()
aef_het = aef_sd.mean(axis=1).rename("aef_heterogeneity").reset_index()
aef_n = grp.size().rename("aef_n_adc").reset_index()
feat = feat.merge(aef_het, on="muncode", how="left").merge(aef_n, on="muncode", how="left")

# ---------- Assemble ----------
D = targets.merge(feat, on="muncode", how="inner")
D["log_n_adc"]        = np.log(D["n_adc_geo"].clip(lower=1))
D["log_mean_ha_adc"]  = np.log(D["mean_ha_adc"].clip(lower=1e-3))
D["log_siap_sd"]      = np.log(D["siap_sd_yield"].clip(lower=1e-3))
FEATURES = ["log_n_adc","irrig_share","maize_share","log_mean_ha_adc",
            "sd_pred","mean_pred","log_siap_sd","siap_mean_yield","aef_heterogeneity"]
D = D.dropna(subset=FEATURES).copy()
print(f"\nModeling frame: {len(D):,} muns x {len(FEATURES)} features")
print("Feature availability OK. Correlations with within_r2_mun:")
for f in FEATURES:
    print(f"  {f:20s}  r={D[[f,'within_r2_mun']].corr().iloc[0,1]:+.3f}  "
          f"rho={D[[f,'within_r2_mun']].corr(method='spearman').iloc[0,1]:+.3f}")

# ---------- Models ----------
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

def standardize(X):
    return (X - X.mean())/X.std()

results = {}
for tgt, clip in [("within_r2_mun",(-2,1)), ("total_contrib",(-2,1)), ("mun_mean_abserr",(0,5))]:
    sub = D.dropna(subset=[tgt]).copy()
    y = sub[tgt].clip(*clip)
    Xz = standardize(sub[FEATURES])
    # OLS
    ols = sm.OLS(y, sm.add_constant(Xz)).fit(cov_type="HC1")
    # RF with 80/20 holdout
    Xtr,Xte,ytr,yte = train_test_split(sub[FEATURES], y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(n_estimators=400, max_depth=8, min_samples_leaf=10,
                                 random_state=42, n_jobs=-1, oob_score=True)
    rf.fit(Xtr, ytr)
    test_r2 = rf.score(Xte, yte)
    perm = permutation_importance(rf, Xte, yte, n_repeats=15, random_state=42, n_jobs=-1)
    results[tgt] = {"ols":ols, "rf":rf, "oob":rf.oob_score_, "test_r2":test_r2,
                     "perm":perm, "ols_r2":ols.rsquared, "Xte":Xte, "yte":yte}
    print(f"\n=== Target: {tgt} (N={len(sub):,}) ===")
    print(f"  OLS R^2 = {ols.rsquared:.3f}   RF OOB R^2 = {rf.oob_score_:.3f}   RF holdout R^2 = {test_r2:.3f}")
    imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
    print("  RF permutation importance (top):")
    for f,v in imp.head(6).items():
        print(f"    {f:20s} {v:+.4f}")

# ---------- Plot: feature importance + holdout calibration ----------
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
tgt_labels = {"within_r2_mun":"Within-mun R^2","total_contrib":"Total-R^2 contribution",
               "mun_mean_abserr":"Mun-mean abs error"}
for j,(tgt,lab) in enumerate(tgt_labels.items()):
    R = results[tgt]
    # top row: permutation importance
    ax = axes[0,j]
    imp = pd.Series(R["perm"].importances_mean, index=FEATURES).sort_values()
    ax.barh(range(len(imp)), imp.values, color="#3a78b0", edgecolor="#222", linewidth=0.3)
    ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp.index, fontsize=8)
    ax.set_xlabel("Permutation importance")
    ax.set_title(f"{lab}\nRF holdout R^2 = {R['test_r2']:.3f}")
    # bottom row: predicted vs actual on holdout
    ax = axes[1,j]
    yhat = R["rf"].predict(R["Xte"])
    ax.scatter(R["yte"], yhat, s=14, alpha=0.4, color="#3a78b0", edgecolor="none")
    lo = min(R["yte"].min(), yhat.min()); hi = max(R["yte"].max(), yhat.max())
    ax.plot([lo,hi],[lo,hi],"--",color="#c44e52",lw=1)
    ax.set_xlabel(f"Actual {lab}"); ax.set_ylabel(f"Predicted {lab}")
    ax.set_title(f"Holdout: predicted vs actual")
plt.suptitle("Ex-ante (mun-level, no ground-truth) prediction of farm-level accuracy", y=1.01)
plt.tight_layout()
fig_path = os.path.join(out_dir, "fig_exante_accuracy_predictors.png")
fig.savefig(fig_path, bbox_inches="tight", dpi=170)
plt.close(fig)
print(f"\nSaved {fig_path}")

# Save feature/importance tables
imp_df = pd.DataFrame({tgt: pd.Series(results[tgt]["perm"].importances_mean, index=FEATURES)
                        for tgt in tgt_labels})
imp_df.to_csv(os.path.join(out_dir, "exante_feature_importance.csv"))
D.to_csv(os.path.join(out_dir, "exante_feature_table.csv"), index=False)
print("Saved exante_feature_importance.csv and exante_feature_table.csv")

# OLS coefficient summary (standardized) for within_r2
print("\nStandardized OLS coefficients for within_r2_mun (HC1):")
ols = results["within_r2_mun"]["ols"]
coef = pd.DataFrame({"coef":ols.params, "se":ols.bse, "t":ols.tvalues, "p":ols.pvalues})
print(coef.to_string(float_format=lambda x: f"{x:+.4f}"))


Targets for 1,855 muns


Loading AEF embeddings (2022)...


  AEF 2022: 126,335 ADCs in 0.0 min

Modeling frame: 1,839 muns x 9 features
Feature availability OK. Correlations with within_r2_mun:
  log_n_adc             r=-0.005  rho=+0.378
  irrig_share           r=-0.004  rho=+0.227
  maize_share           r=+0.045  rho=-0.140
  log_mean_ha_adc       r=-0.024  rho=-0.283
  sd_pred               r=-0.013  rho=+0.310
  mean_pred             r=-0.002  rho=+0.261
  log_siap_sd           r=-0.007  rho=+0.261
  siap_mean_yield       r=+0.004  rho=+0.275
  aef_heterogeneity     r=+0.013  rho=+0.335



=== Target: within_r2_mun (N=1,830) ===
  OLS R^2 = 0.178   RF OOB R^2 = 0.267   RF holdout R^2 = 0.211
  RF permutation importance (top):
    siap_mean_yield      +0.1152
    mean_pred            +0.1066
    log_n_adc            +0.0734
    irrig_share          +0.0480
    log_mean_ha_adc      +0.0461
    sd_pred              +0.0335



=== Target: total_contrib (N=1,839) ===
  OLS R^2 = 0.186   RF OOB R^2 = 0.530   RF holdout R^2 = 0.491
  RF permutation importance (top):
    mean_pred            +0.4728
    siap_mean_yield      +0.3756
    sd_pred              +0.0729
    maize_share          +0.0180
    irrig_share          +0.0151
    log_n_adc            +0.0069



=== Target: mun_mean_abserr (N=1,839) ===
  OLS R^2 = 0.275   RF OOB R^2 = 0.294   RF holdout R^2 = 0.445
  RF permutation importance (top):
    siap_mean_yield      +0.2468
    mean_pred            +0.1501
    log_n_adc            +0.0577
    log_mean_ha_adc      +0.0529
    irrig_share          +0.0249
    log_siap_sd          +0.0215



Saved /home/jdesktop/Dropbox/Projects/Maize_prediction/plots/coauthor_extras_paper/fig_exante_accuracy_predictors.png
Saved exante_feature_importance.csv and exante_feature_table.csv

Standardized OLS coefficients for within_r2_mun (HC1):
                     coef      se        t       p
const             -1.1560 +0.0191 -60.5417 +0.0000
log_n_adc         +0.1267 +0.0282  +4.4893 +0.0000
irrig_share       +0.0934 +0.0228  +4.0869 +0.0000
maize_share       -0.0081 +0.0210  -0.3843 +0.7008
log_mean_ha_adc   -0.1152 +0.0231  -4.9862 +0.0000
sd_pred           +0.1991 +0.0365  +5.4517 +0.0000
mean_pred         -0.3340 +0.0529  -6.3134 +0.0000
log_siap_sd       +0.0836 +0.0310  +2.6962 +0.0070
siap_mean_yield   +0.0405 +0.0479  +0.8448 +0.3982
aef_heterogeneity +0.1163 +0.0254  +4.5729 +0.0000


#### B. Enriched within-R² model + trust index (16 features)

Source: `exante_enriched_model.py` (patched: float-cast AEF, NaN-safe covariance,
drops the inline map — see Section C). Adds 7 within-heterogeneity features and
writes the CV-predicted within-R² **trust index** (`exante_trust_index.csv`).
Run in `ml_cuda`.

*(The earlier `exante_enriched.py` combined this modeling with the map in one
script; it is fully superseded by this cell plus Section C.)*


In [2]:
"""Enriched ex-ante prediction of within-mun R^2 + trust-index map.

Adds within-heterogeneity features (the mechanism behind within-R^2):
  irrig_share_sd  : within-mun SD of ADC irrigation share (irrigation gradient -> yield variance)
  frac_irrigated  : fraction of ADCs in mun that are irrigated (>5% irrig)
  adc_area_cv     : CV of ADC areas within mun (land-unit heterogeneity)
  maizeland_sd    : within-mun SD of maize-land share
  aef_spread      : sqrt(trace of within-mun AEF covariance) = total satellite spread
  aef_pc1_frac    : lambda_max / trace (is within-mun variation a single dominant gradient?)
  aef_eff_dim     : participation ratio (Sum l)^2 / Sum l^2 (effective # varying AEF dims)
"""
import os, time
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams.update({"font.family":"serif","figure.dpi":130,
                     "axes.titlesize":12,"axes.labelsize":11,
                     "xtick.labelsize":9,"ytick.labelsize":9,"legend.fontsize":9})

home = os.path.expanduser("~")
proj = os.path.join(home, "Dropbox/Projects/Maize_prediction")
data = os.path.join(proj, "Data")
preds_dir = os.path.join(data, "predictions")
out_dir = os.path.join(proj, "plots/coauthor_extras_paper")
siap_pkg = os.path.join(home, "Dropbox/Projects/The Promise of Crop Substitution/data/SIAP/Cleaned")

def r2(y, yh):
    m = np.isfinite(y) & np.isfinite(yh)
    y, yh = np.asarray(y[m]), np.asarray(yh[m])
    if len(y) < 2: return np.nan
    st = np.sum((y - y.mean())**2)
    return 1 - np.sum((y - yh)**2)/st if st > 0 else np.nan

# ---------- Targets ----------
df = pd.read_parquet(os.path.join(preds_dir, "adc_aef_hist_ens_eval.parquet"))
df = df[["muncode","adc","yield","pred"]].dropna(subset=["yield","pred"]).copy()
df["muncode"] = df["muncode"].astype(str).str.zfill(5)
gmean = df["yield"].mean()
rows=[]
for mun,g in df.groupby("muncode"):
    if len(g)<2: continue
    ym,pm=g["yield"].mean(),g["pred"].mean()
    ssr=float(((g["yield"]-g["pred"])**2).sum())
    sstw=float(((g["yield"]-ym)**2).sum()); sstg=float(((g["yield"]-gmean)**2).sum())
    rows.append({"muncode":mun,"within_r2_mun":1-ssr/sstw if sstw>0 else np.nan,
                  "total_contrib":1-ssr/sstg if sstg>0 else np.nan,
                  "mun_mean_abserr":abs(pm-ym)})
targets=pd.DataFrame(rows)

# ---------- Baseline features from drivers_mun ----------
dm=pd.read_csv(os.path.join(out_dir,"drivers_mun.csv"))
dm["muncode"]=dm["muncode"].astype(str).str.zfill(5)
feat=dm[["muncode","irrig_share","maize_share","sd_pred","mean_pred"]].copy()

# SIAP cross-year
siap=pd.read_stata(os.path.join(siap_pkg,"siap_ag_prod_estimation_ca2007.dta"),
                    columns=["year","muncode","name","q","ha_planted"])
siap=siap[(siap["name"]=="Maize")&siap["year"].between(2007,2024)].copy()
siap["yld"]=siap["q"]/siap["ha_planted"].replace(0,np.nan); siap=siap.dropna(subset=["yld"])
siap["muncode"]=siap["muncode"].astype(str).str.zfill(5)
sa=siap.groupby("muncode").agg(siap_sd_yield=("yld","std"),siap_mean_yield=("yld","mean"),
                                 nyr=("year","nunique")).reset_index()
sa=sa[sa["nyr"]>=5]
feat=feat.merge(sa[["muncode","siap_sd_yield","siap_mean_yield"]],on="muncode",how="left")

# ---------- NEW: ADC-level irrigation / area / maize heterogeneity ----------
ag=pd.read_csv(os.path.join(data,"SIAP_agland/Output/2007_adcs_agland_area.csv"))
ag["muncode"]=ag["adc07"].astype(str).str.replace("-","",regex=False).str[:5]
ag["irr_sh"]=ag["siap_irrig_area"]/ag["siap_agland_area"].replace(0,np.nan)
ag["maize_sh"]=ag["maize_land"]/ag["siap_agland_area"].replace(0,np.nan)
het=ag.groupby("muncode").agg(
    irrig_share_sd=("irr_sh","std"),
    frac_irrigated=("irr_sh", lambda s: float((s>0.05).mean())),
    adc_area_sd=("adc_area","std"), adc_area_mean=("adc_area","mean"),
    n_adc_geo=("adc_area","size"), mean_ha_adc=("adc_area","mean"),
    maizeland_sd=("maize_sh","std")).reset_index()
het["adc_area_cv"]=het["adc_area_sd"]/het["adc_area_mean"].replace(0,np.nan)
feat=feat.merge(het[["muncode","irrig_share_sd","frac_irrigated","adc_area_cv","maizeland_sd","n_adc_geo","mean_ha_adc"]],
                 on="muncode",how="left")

# ---------- NEW: AEF covariance spread metrics (2022) ----------
print("Loading AEF 2022 + computing per-mun spread metrics...")
t0=time.time()
aef=pd.read_parquet(os.path.join(data,"alpha_earth/alpha_earth_mex_adcs.parquet"),
                     filters=[("year","==",2022)])
acols=[f"A{i:02d}" for i in range(64)]
aef["muncode"]=aef["adcid"].astype(str).str.replace("-","",regex=False).str[:5]
het_rows=[]
for mun,g in aef.groupby("muncode"):
    if len(g)<3:
        het_rows.append({"muncode":mun,"aef_heterogeneity":np.nan,"aef_spread":np.nan,
                          "aef_pc1_frac":np.nan,"aef_eff_dim":np.nan}); continue
    X=g[acols].to_numpy(dtype=float)
    X=X[np.isfinite(X).all(axis=1)]
    if len(X)<3:
        het_rows.append({"muncode":mun,"aef_heterogeneity":np.nan,"aef_spread":np.nan,
                          "aef_pc1_frac":np.nan,"aef_eff_dim":np.nan}); continue
    perdim_var=X.var(axis=0)
    tr=float(perdim_var.sum())            # trace = total within-mun variance (no eig needed)
    pc1_frac=np.nan; eff_dim=np.nan
    try:
        C=np.cov(X, rowvar=False)
        C=np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)
        ev=np.linalg.eigvalsh(C); ev=ev[ev>1e-12]
        if ev.size and ev.sum()>0:
            pc1_frac=float(ev.max()/ev.sum())
            eff_dim=float((ev.sum()**2)/np.sum(ev**2))
    except np.linalg.LinAlgError:
        pass
    het_rows.append({"muncode":mun,
        "aef_heterogeneity":float(np.sqrt(perdim_var).mean()),
        "aef_spread":float(np.sqrt(tr)) if tr>0 else np.nan,
        "aef_pc1_frac":pc1_frac,"aef_eff_dim":eff_dim})
aef_het=pd.DataFrame(het_rows)
feat=feat.merge(aef_het,on="muncode",how="left")
print(f"  done in {(time.time()-t0)/60:.1f} min")

# ---------- Assemble + transforms ----------
D=targets.merge(feat,on="muncode",how="inner")
D["log_n_adc"]=np.log(D["n_adc_geo"].clip(lower=1))
D["log_mean_ha_adc"]=np.log(D["mean_ha_adc"].clip(lower=1e-3))
D["log_siap_sd"]=np.log(D["siap_sd_yield"].clip(lower=1e-3))

BASELINE=["log_n_adc","irrig_share","maize_share","log_mean_ha_adc","sd_pred","mean_pred",
          "log_siap_sd","siap_mean_yield","aef_heterogeneity"]
NEW=["irrig_share_sd","frac_irrigated","adc_area_cv","maizeland_sd",
     "aef_spread","aef_pc1_frac","aef_eff_dim"]
ENRICHED=BASELINE+NEW

D=D.dropna(subset=ENRICHED+["within_r2_mun"]).copy()
print(f"\nModeling frame: {len(D):,} muns; baseline {len(BASELINE)} + new {len(NEW)} features")

# Spearman with within_r2 for NEW features
print("\nSpearman rho of NEW features with within_r2_mun:")
for f in NEW:
    print(f"  {f:18s} rho={D[[f,'within_r2_mun']].corr(method='spearman').iloc[0,1]:+.3f}")

# ---------- Models: baseline vs enriched for within_r2 ----------
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

y=D["within_r2_mun"].clip(-2,1)
def eval_set(features, label):
    Xtr,Xte,ytr,yte=train_test_split(D[features],y,test_size=0.2,random_state=42)
    rf=RandomForestRegressor(n_estimators=500,max_depth=10,min_samples_leaf=8,
                              random_state=42,n_jobs=-1,oob_score=True)
    rf.fit(Xtr,ytr)
    print(f"  {label:18s} OOB R^2={rf.oob_score_:.3f}  holdout R^2={rf.score(Xte,yte):.3f}")
    return rf,Xte,yte
print("\n=== within_r2_mun predictability: baseline vs enriched ===")
rf_b,_,_=eval_set(BASELINE,"baseline (9)")
rf_e,Xte,yte=eval_set(ENRICHED,"enriched (16)")

perm=permutation_importance(rf_e,Xte,yte,n_repeats=20,random_state=42,n_jobs=-1)
imp=pd.Series(perm.importances_mean,index=ENRICHED).sort_values(ascending=False)
print("\nEnriched within_r2 permutation importance:")
for f,v in imp.items(): print(f"  {f:18s} {v:+.4f}")
imp.to_csv(os.path.join(out_dir,"exante_within_importance_enriched.csv"))

# ---------- Trust index (a): CV-predicted within_r2 across all muns ----------
rf_full=RandomForestRegressor(n_estimators=500,max_depth=10,min_samples_leaf=8,
                               random_state=42,n_jobs=-1)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
D["trust_within"]=cross_val_predict(rf_full,D[ENRICHED],y,cv=kf,n_jobs=-1)
print(f"\nTrust index (CV-predicted within_r2): corr with actual = "
      f"{np.corrcoef(D['trust_within'], y)[0,1]:+.3f}")
D[["muncode","within_r2_mun","trust_within"]+ENRICHED].to_csv(
    os.path.join(out_dir,"exante_trust_index.csv"),index=False)

# ---------- Plots ----------
# 1) importance + baseline-vs-enriched holdout
fig,axes=plt.subplots(1,2,figsize=(15,6))
ax=axes[0]
imp_sorted=imp.sort_values()
colors=["#c44e52" if f in NEW else "#3a78b0" for f in imp_sorted.index]
ax.barh(range(len(imp_sorted)),imp_sorted.values,color=colors,edgecolor="#222",linewidth=0.3)
ax.set_yticks(range(len(imp_sorted))); ax.set_yticklabels(imp_sorted.index,fontsize=8.5)
ax.set_xlabel("Permutation importance (enriched RF, within-R^2)")
ax.set_title("Ex-ante drivers of within-mun R^2\n(red = new within-heterogeneity features)")
ax=axes[1]
yhat=rf_e.predict(Xte)
ax.scatter(yte,yhat,s=16,alpha=0.4,color="#3a78b0",edgecolor="none")
ax.plot([-2,1],[-2,1],"--",color="#c44e52",lw=1)
ax.set_xlabel("Actual within-mun R^2 (clipped)"); ax.set_ylabel("Predicted within-mun R^2")
ax.set_title(f"Enriched holdout: R^2 = {rf_e.score(Xte,yte):.3f}")
plt.tight_layout()
fig.savefig(os.path.join(out_dir,"fig_exante_within_enriched.png"),bbox_inches="tight",dpi=170)
plt.close(fig)
print("Saved fig_exante_within_enriched.png")

print("Modeling done; trust index + importances saved. Map handled separately.")


Loading AEF 2022 + computing per-mun spread metrics...


  done in 0.0 min

Modeling frame: 1,738 muns; baseline 9 + new 7 features

Spearman rho of NEW features with within_r2_mun:
  irrig_share_sd     rho=+0.246
  frac_irrigated     rho=+0.200
  adc_area_cv        rho=+0.165
  maizeland_sd       rho=+0.014
  aef_spread         rho=+0.294
  aef_pc1_frac       rho=-0.198
  aef_eff_dim        rho=+0.240

=== within_r2_mun predictability: baseline vs enriched ===


  baseline (9)       OOB R^2=0.245  holdout R^2=0.190


  enriched (16)      OOB R^2=0.240  holdout R^2=0.199



Enriched within_r2 permutation importance:
  siap_mean_yield    +0.0977
  mean_pred          +0.0651
  irrig_share_sd     +0.0504
  sd_pred            +0.0322
  log_mean_ha_adc    +0.0315
  log_n_adc          +0.0199
  log_siap_sd        +0.0092
  aef_heterogeneity  +0.0067
  frac_irrigated     +0.0055
  maize_share        +0.0032
  irrig_share        +0.0024
  maizeland_sd       +0.0021
  adc_area_cv        +0.0021
  aef_spread         +0.0013
  aef_eff_dim        -0.0029
  aef_pc1_frac       -0.0034



Trust index (CV-predicted within_r2): corr with actual = +0.483


Saved fig_exante_within_enriched.png
Modeling done; trust index + importances saved. Map handled separately.


#### C. Trust-index choropleth

Source: `trust_map.py`. Ex-ante predicted vs actual within-mun R² maps from
`exante_trust_index.csv`. **Run in `mpc_env`** (needs geopandas).


In [3]:
"""Trust-index choropleth: ex-ante predicted vs actual within-mun R^2."""
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
mpl.rcParams.update({"font.family":"serif","figure.dpi":130})

home=os.path.expanduser("~")
data=os.path.join(home,"Dropbox/Projects/Maize_prediction/Data")
out_dir=os.path.join(home,"Dropbox/Projects/Maize_prediction/plots/coauthor_extras_paper")

D=pd.read_csv(os.path.join(out_dir,"exante_trust_index.csv"),dtype={"muncode":str})
D["muncode"]=D["muncode"].str.zfill(5)

shp=os.path.join(data,"SIAP_agland/Output/agland_mun16.shp")
mun=gpd.read_file(shp).dissolve(by="muncode",as_index=False)[["muncode","geometry"]]
mun=mun.merge(D[["muncode","trust_within","within_r2_mun"]],on="muncode",how="left")

states_shp=os.path.join(home,"Dropbox/Projects/Crop_misallocation/Data/Municipality_shp/STATES.shp")
states=gpd.read_file(states_shp).to_crs(mun.crs) if os.path.exists(states_shp) else None

fig,axes=plt.subplots(1,2,figsize=(18,8))
norm=TwoSlopeNorm(vmin=-1,vcenter=0,vmax=0.6)
for ax,col,lab in [(axes[0],"trust_within","Ex-ante predicted within-R² (trust index)"),
                    (axes[1],"within_r2_mun","Actual within-R² (INEGI 2022)")]:
    mun.plot(ax=ax,column=col,cmap="RdYlBu",norm=norm,edgecolor="none",
              legend=True,legend_kwds={"label":lab,"shrink":0.6},
              missing_kwds={"color":"#eeeeee","label":"no data"})
    if states is not None: states.boundary.plot(ax=ax,color="black",linewidth=0.3)
    ax.set_title(lab); ax.set_axis_off()
plt.suptitle("Model-trust index: ex-ante predicted vs actual within-mun R²  "
              f"(CV corr = {np.corrcoef(D['trust_within'].dropna(), D.loc[D['trust_within'].notna(),'within_r2_mun'].clip(-2,1))[0,1]:+.3f})",
              y=1.0)
plt.tight_layout()
fp=os.path.join(out_dir,"fig_trust_index_map.png")
fig.savefig(fp,bbox_inches="tight",dpi=170)
plt.close(fig)
print(f"Saved {fp}")
print(f"Muns mapped: {mun['trust_within'].notna().sum()}")


ERROR 1: PROJ: proj_create_from_database: Open of /home/jdesktop/miniforge3/envs/geo_env/share/proj failed


Saved /home/jdesktop/Dropbox/Projects/Maize_prediction/plots/coauthor_extras_paper/fig_trust_index_map.png
Muns mapped: 1735


#### D. Push — target-noise diagnostic, robust targets, multi-year AEF, interactions

Source: `exante_push.py`. Key finding: the ~0.20 "ceiling" on within-skill
predictability was largely **target noise**. On well-estimated muns the
per-mun **Spearman-ρ** predictor reaches holdout R² ≈ 0.44 and the
**positive-skill classifier** reaches AUC ≈ 0.81; `irrig_share_sd × log_n_adc`
is significant (p = 0.001). Run in `ml_cuda`.


In [4]:
"""Push ex-ante within-skill predictability without new data:
1. Sample-size diagnostic (maize-obs n_eval thresholds) + precision weighting
2. Robust targets: per-mun Spearman rho + binary P(within-R^2>0) classifier (AUC)
3. Multi-year AEF spread (2017-2024 mean + temporal stability)
4. Interpretable interaction terms in OLS
"""
import os, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams.update({"font.family":"serif","figure.dpi":130,
                     "axes.titlesize":12,"axes.labelsize":11,
                     "xtick.labelsize":9,"ytick.labelsize":9,"legend.fontsize":9})

home=os.path.expanduser("~")
proj=os.path.join(home,"Dropbox/Projects/Maize_prediction")
data=os.path.join(proj,"Data"); preds_dir=os.path.join(data,"predictions")
out_dir=os.path.join(proj,"plots/coauthor_extras_paper")

def r2(y,yh):
    m=np.isfinite(y)&np.isfinite(yh); y,yh=np.asarray(y[m]),np.asarray(yh[m])
    if len(y)<2: return np.nan
    st=np.sum((y-y.mean())**2); return 1-np.sum((y-yh)**2)/st if st>0 else np.nan

# ---- Base feature table (16 ex-ante features already computed) ----
D=pd.read_csv(os.path.join(out_dir,"exante_trust_index.csv"),dtype={"muncode":str})
D["muncode"]=D["muncode"].str.zfill(5)
D["n_adc"]=np.exp(D["log_n_adc"]).round().astype(int)
ENRICHED=["log_n_adc","irrig_share","maize_share","log_mean_ha_adc","sd_pred","mean_pred",
          "log_siap_sd","siap_mean_yield","aef_heterogeneity","irrig_share_sd","frac_irrigated",
          "adc_area_cv","maizeland_sd","aef_spread","aef_pc1_frac","aef_eff_dim"]

# ---- Targets: within_r2 (have), Spearman rho, binary positive-skill ----
ev=pd.read_parquet(os.path.join(preds_dir,"adc_aef_hist_ens_eval.parquet"))
ev=ev[["muncode","yield","pred"]].dropna(subset=["yield","pred"]).copy()
ev["muncode"]=ev["muncode"].astype(str).str.zfill(5)
# n_eval = # ADCs per mun WITH maize ground truth -> target-reliability gate
# (distinct from the ex-ante FEATURE log_n_adc, which counts ALL ADCs by geography)
D=D.merge(ev.groupby("muncode").size().rename("n_eval").reset_index(),on="muncode",how="left")
rho_rows=[]
for mun,g in ev.groupby("muncode"):
    if len(g)<5: continue
    rho,_=spearmanr(g["yield"],g["pred"])
    rho_rows.append({"muncode":mun,"rho_mun":rho})
D=D.merge(pd.DataFrame(rho_rows),on="muncode",how="left")
D["pos_skill"]=(D["within_r2_mun"]>0).astype(int)

# ---- (3) Multi-year AEF spread (2017-2024) ----
print("Computing multi-year AEF spread (2017-2024)...")
t0=time.time()
acols=[f"A{i:02d}" for i in range(64)]
aef=pd.read_parquet(os.path.join(data,"alpha_earth/alpha_earth_mex_adcs.parquet"),
                     columns=acols+["adcid","year"])
aef["muncode"]=aef["adcid"].astype(str).str.replace("-","",regex=False).str[:5]
# per (mun,year): spread = sqrt(sum of per-dim variance)
def spread_of(block):
    X=block[acols].to_numpy(dtype=float); X=X[np.isfinite(X).all(axis=1)]
    if len(X)<3: return np.nan
    return float(np.sqrt(X.var(axis=0).sum()))
sp=(aef.groupby(["muncode","year"]).apply(spread_of, include_groups=False)
       .rename("spread_yr").reset_index())
my=sp.groupby("muncode")["spread_yr"].agg(aef_spread_my="mean", aef_spread_stab="std").reset_index()
D=D.merge(my,on="muncode",how="left")
print(f"  done {(time.time()-t0)/60:.1f} min; multiyear spread for {D['aef_spread_my'].notna().sum()} muns")

FEAT_MY=ENRICHED+["aef_spread_my","aef_spread_stab"]
Dm=D.dropna(subset=FEAT_MY+["within_r2_mun"]).copy()
print(f"Frame with multi-year features: {len(Dm):,} muns")

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_predict, KFold, cross_val_score
from sklearn.metrics import roc_auc_score

def rf_holdout(df, feats, target, clip=None, weight=None, seed=42):
    sub=df.dropna(subset=feats+[target]).copy()
    y=sub[target].clip(*clip) if clip else sub[target]
    idx=np.arange(len(sub))
    tr,te=train_test_split(idx,test_size=0.2,random_state=seed)
    rf=RandomForestRegressor(n_estimators=500,max_depth=10,min_samples_leaf=8,
                              random_state=seed,n_jobs=-1)
    w=sub[weight].to_numpy()[tr] if weight else None
    rf.fit(sub[feats].iloc[tr],y.iloc[tr],sample_weight=w)
    return rf.score(sub[feats].iloc[te],y.iloc[te]), len(sub)

print("\n=== (1) Sample-size diagnostic: within-R^2 holdout RF R^2 ===")
for thr in [2,10,20,30,50]:
    sub=Dm[Dm["n_eval"]>=thr]
    r2h,n=rf_holdout(sub,FEAT_MY,"within_r2_mun",clip=(-2,1))
    print(f"  n_eval>={thr:<3d}  N={n:<5d}  holdout R^2={r2h:+.3f}")
print("  precision-weighted (all muns, weight=n_eval):")
r2w,n=rf_holdout(Dm,FEAT_MY,"within_r2_mun",clip=(-2,1),weight="n_eval")
print(f"    N={n}  holdout R^2={r2w:+.3f}")

print("\n=== (2) Robust targets ===")
# Spearman rho regression
sub=Dm.dropna(subset=["rho_mun"])
r2_rho,nr=rf_holdout(sub,FEAT_MY,"rho_mun")
print(f"  Per-mun Spearman rho  N={nr}  holdout R^2={r2_rho:+.3f}")
for thr in [20,30]:
    s2=sub[sub["n_eval"]>=thr]
    r2_rt,n2=rf_holdout(s2,FEAT_MY,"rho_mun")
    print(f"    rho, n_eval>={thr}  N={n2}  holdout R^2={r2_rt:+.3f}")
# Binary classifier P(within-R^2>0)
sub=Dm.dropna(subset=["pos_skill"])
tr,te=train_test_split(np.arange(len(sub)),test_size=0.2,random_state=42,stratify=sub["pos_skill"])
clf=RandomForestClassifier(n_estimators=500,max_depth=10,min_samples_leaf=8,random_state=42,n_jobs=-1)
clf.fit(sub[FEAT_MY].iloc[tr],sub["pos_skill"].iloc[tr])
auc=roc_auc_score(sub["pos_skill"].iloc[te],clf.predict_proba(sub[FEAT_MY].iloc[te])[:,1])
print(f"  Binary P(within-R^2>0)  N={len(sub)}  base rate={sub['pos_skill'].mean():.2f}  holdout AUC={auc:.3f}")
for thr in [20,30]:
    s2=sub[sub["n_eval"]>=thr]
    tr2,te2=train_test_split(np.arange(len(s2)),test_size=0.2,random_state=42,stratify=s2["pos_skill"])
    c2=RandomForestClassifier(n_estimators=500,max_depth=10,min_samples_leaf=8,random_state=42,n_jobs=-1)
    c2.fit(s2[FEAT_MY].iloc[tr2],s2["pos_skill"].iloc[tr2])
    auc2=roc_auc_score(s2["pos_skill"].iloc[te2],c2.predict_proba(s2[FEAT_MY].iloc[te2])[:,1])
    print(f"    AUC n_eval>={thr}  N={len(s2)}  base rate={s2['pos_skill'].mean():.2f}  AUC={auc2:.3f}")

print("\n=== (3) Does multi-year AEF spread help? (within-R^2, n_eval>=20) ===")
base=Dm[Dm["n_eval"]>=20]
r2_no,_=rf_holdout(base,ENRICHED,"within_r2_mun",clip=(-2,1))
r2_my,_=rf_holdout(base,FEAT_MY,"within_r2_mun",clip=(-2,1))
print(f"  without multi-year AEF: R^2={r2_no:+.3f}")
print(f"  with    multi-year AEF: R^2={r2_my:+.3f}")
print(f"  rho(aef_spread_my, within_r2)={base[['aef_spread_my','within_r2_mun']].corr(method='spearman').iloc[0,1]:+.3f}")
print(f"  rho(aef_spread_stab, within_r2)={base[['aef_spread_stab','within_r2_mun']].corr(method='spearman').iloc[0,1]:+.3f}")

print("\n=== (4) Interpretable interactions (OLS on n_eval>=20, within_r2 clipped) ===")
import statsmodels.api as sm
sub=Dm[Dm["n_eval"]>=20].dropna(subset=FEAT_MY+["within_r2_mun"]).copy()
core=["log_n_adc","irrig_share_sd","aef_spread","log_siap_sd","siap_mean_yield","mean_pred"]
Z=(sub[core]-sub[core].mean())/sub[core].std()
Z["irrSD_x_n"]   = Z["irrig_share_sd"]*Z["log_n_adc"]
Z["spread_x_sd"] = Z["aef_spread"]*Z["log_siap_sd"]
y=sub["within_r2_mun"].clip(-2,1)
m_main=sm.OLS(y,sm.add_constant(Z[core])).fit(cov_type="HC1")
m_int =sm.OLS(y,sm.add_constant(Z[core+["irrSD_x_n","spread_x_sd"]])).fit(cov_type="HC1")
print(f"  main-effects R^2={m_main.rsquared:.3f}   with interactions R^2={m_int.rsquared:.3f}")
print("  interaction coefs (HC1):")
for k in ["irrSD_x_n","spread_x_sd"]:
    print(f"    {k:14s} coef={m_int.params[k]:+.4f}  t={m_int.tvalues[k]:+.2f}  p={m_int.pvalues[k]:.3f}")

# Save augmented table
D.to_csv(os.path.join(out_dir,"exante_trust_index_v2.csv"),index=False)
print("\nSaved exante_trust_index_v2.csv (adds rho_mun, pos_skill, multi-year AEF spread)")

# ---- Plot: predictability vs sample-size threshold ----
fig,ax=plt.subplots(figsize=(9,5.5))
thrs=[2,10,20,30,50]
wr2=[rf_holdout(Dm[Dm["n_eval"]>=t],FEAT_MY,"within_r2_mun",clip=(-2,1))[0] for t in thrs]
rho=[rf_holdout(Dm[(Dm["n_eval"]>=t)].dropna(subset=["rho_mun"]),FEAT_MY,"rho_mun")[0] for t in thrs]
ax.plot(thrs,wr2,"-o",color="#3a78b0",lw=2,ms=8,label="Predict within-R²")
ax.plot(thrs,rho,"-s",color="#c44e52",lw=2,ms=8,label="Predict Spearman ρ")
ax.set_xlabel("Minimum ADCs per mun (target estimation quality →)")
ax.set_ylabel("Holdout R² of the ex-ante predictor")
ax.set_title("Is within-skill unpredictable, or is the target noisy?\nEx-ante predictability rises as the per-mun target is better-estimated")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(out_dir,"fig_exante_samplesize_diagnostic.png"),bbox_inches="tight",dpi=170)
plt.close(fig)
print("Saved fig_exante_samplesize_diagnostic.png")


Computing multi-year AEF spread (2017-2024)...


  done 0.1 min; multiyear spread for 1738 muns
Frame with multi-year features: 1,738 muns

=== (1) Sample-size diagnostic: within-R^2 holdout RF R^2 ===


  n_eval>=2    N=1738   holdout R^2=+0.204


  n_eval>=10   N=1264   holdout R^2=+0.234


  n_eval>=20   N=967    holdout R^2=+0.287


...

  Binary P(within-R^2>0)  N=1738  base rate=0.14  holdout AUC=0.772


    AUC n_eval>=20  N=967  base rate=0.17  AUC=0.809


    AUC n_eval>=30  N=765  base rate=0.18  AUC=0.770

=== (3) Does multi-year AEF spread help? (within-R^2, n_eval>=20) ===


  without multi-year AEF: R^2=+0.266
  with    multi-year AEF: R^2=+0.287
  rho(aef_spread_my, within_r2)=+0.148
  rho(aef_spread_stab, within_r2)=+0.164

=== (4) Interpretable interactions (OLS on n_eval>=20, within_r2 clipped) ===
  main-effects R^2=0.099   with interactions R^2=0.112
  interaction coefs (HC1):
    irrSD_x_n      coef=+0.1045  t=+3.83  p=0.000
    spread_x_sd    coef=-0.0245  t=-0.88  p=0.381

Saved exante_trust_index_v2.csv (adds rho_mun, pos_skill, multi-year AEF spread)


Saved fig_exante_samplesize_diagnostic.png
